# Phase 1 — Architecture 2: Encoder-Decoder Transformer (v3 tokenizer)

Same encoder-decoder architecture as before (6+6 layers, d_model 768, 12 heads, FFN 3072, src 1024 / tgt 512,
tied decoder embedding + LM head). What changed:

* **Data**: reads the `uint16` token bins from `updated-tokenize-data` (`<source>_100m.bin`), not the old `uint32` files.
* **Tokenizer**: loads the saved v3 `tokenizer.py` (`load_tokenizer`) — correct JSON-lines merges, byte-exact
  `encode`/`decode`, special IDs `<pad>=0 <unk>=1 <bos>=2 <eos>=3`. The old notebook's hand-written `▁`-based
  encode/decode no longer matches this tokenizer, so it is gone.
* **Speed on Kaggle**: fp16 + GradScaler on T4/P100 (the old run picked bf16, which T4 only emulates — very slow),
  no gradient checkpointing, batch 4 × accum 4 (same effective batch 16), fused AdamW, batches sampled straight from
  RAM (no DataLoader workers), fewer `.item()` syncs, and a time limit that saves and stops before Kaggle's 12 h cutoff.

Add both notebooks (`updated-bpe-tokenizer`, `updated-tokenize-data`) as inputs, pick **GPU T4** (or P100), then Run All.

## At a glance — Phase 1 · Architecture-2 — Transformer encoder-decoder

| | |
|---|---|
| **Codabench** | BERTScore F1 **0.8289** (the best Phase-1 model) |
| **Validation** | loss 1.4996 (perplexity 4.48) on the 512 target tokens after 34,000 steps |
| **Size** | 149.6M parameters, 6 encoder + 6 decoder layers, width 768 |

**In simple words:** two halves. The **encoder** reads 1,024 tokens of legal text, looking in both directions; the **decoder**
then writes the **next 512 tokens**, looking back at what it has written and at the encoder's reading. Later phases mostly use the
encoder half as a compact text reader (and the KD students of Phases 2–5 and 7).

**How it works:** pre-LayerNorm transformer, learned positions, GELU feed-forward 3,072, decoder embedding tied with the output layer;
random 1,536-token windows straight from RAM; float16 + GradScaler, AdamW with warm-up + cosine decay; stops safely before Kaggle's
12-hour limit and resumes from its checkpoint.

### CELL 1 · Master configuration

**In simple words:** Every setting: a debug switch (tiny model, 20 steps), model size, 1,024 tokens in → 512 out, batch, learning rate and a time limit.

In [1]:
# ============================================================
# PHASE 1 — ARCHITECTURE 2 (ENCODER-DECODER)
# CELL 1 — MASTER CONFIGURATION
# ============================================================

import os
import gc
import json
import math
import time
import shutil
import random
import importlib.util
from pathlib import Path

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint


# ------------------------------------------------------------
# 1. DEBUG SWITCH
#    True  -> tiny model + 20 steps, checks the whole pipeline in ~1 min
#    False -> full Architecture-2 run
# ------------------------------------------------------------

DEBUG_MODE = False


# ------------------------------------------------------------
# 2. PATHS (Kaggle inputs = outputs of your two earlier notebooks)
#    If a path is wrong, Cell 2/3 searches /kaggle/input for it.
# ------------------------------------------------------------

TOKENIZER_DIR = Path(
    "/kaggle/input/notebooks/ritikguptatab/updated-bpe-tokenizer/"
    "legal_tokenizer_v3/tokenizer/final_tokenizer"
)

TOKEN_DIR = Path(
    "/kaggle/input/notebooks/ritikguptatab/updated-tokenize-data/"
    "legal_tokenized_100m"
)

OUTPUT_DIR = Path("/kaggle/working/architecture_2_encoder_decoder")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LAST_CHECKPOINT = OUTPUT_DIR / "last_encoder_decoder.pt"   # full state, for resume
BEST_CHECKPOINT = OUTPUT_DIR / "best_encoder_decoder.pt"   # model weights only


# ------------------------------------------------------------
# 3. DATA
# ------------------------------------------------------------

SOURCES = ["acts", "high_courts", "supreme_court", "tribunals"]

TOKEN_FILE_PATTERN = "{source}_100m.bin"     # written by updated-tokenize-data
TOKEN_DTYPE = np.uint16                      # that notebook saved uint16

VOCAB_SIZE = 32004

MIXING_RATIOS = {
    "acts": 0.15,
    "high_courts": 0.35,
    "supreme_court": 0.25,
    "tribunals": 0.25,
}

# Last 1% of every source file is held out for validation.
VAL_FRACTION = 0.02

# Total training-token pool across all sources.
MAX_TRAIN_TOKENS = 800_000_000

# Each source gets its share of the pool from MIXING_RATIOS
# acts 90M | high_courts 210M | supreme_court 150M | tribunals 150M
TRAIN_TOKEN_BUDGET = {
    source: int(MAX_TRAIN_TOKENS * MIXING_RATIOS[source])
    for source in SOURCES
}
# ------------------------------------------------------------
# 4. MODEL (same architecture as before)
# ------------------------------------------------------------

if DEBUG_MODE:
    SRC_LEN, TGT_LEN = 128, 64
    D_MODEL, N_HEADS, N_LAYERS, FFN_DIM = 128, 4, 2, 512
else:
    SRC_LEN, TGT_LEN = 1024, 512
    D_MODEL, N_HEADS, N_LAYERS, FFN_DIM = 768, 12, 6, 3072

DROPOUT = 0.1
ENCODER_LAYERS = N_LAYERS
DECODER_LAYERS = N_LAYERS

# Each training example needs SRC_LEN + TGT_LEN consecutive tokens.
WINDOW_LEN = SRC_LEN + TGT_LEN


# ------------------------------------------------------------
# 5. TRAINING
#    Effective batch = BATCH_SIZE * GRAD_ACCUM_STEPS = 16 (same as before)
#    If you hit CUDA OOM: BATCH_SIZE = 2, GRAD_ACCUM_STEPS = 8
# ------------------------------------------------------------

if DEBUG_MODE:
    BATCH_SIZE, GRAD_ACCUM_STEPS = 2, 2
    MAX_STEPS, WARMUP_STEPS = 20, 5
    EVAL_INTERVAL, SAVE_INTERVAL = 10, 10
    VAL_BATCHES, LOG_INTERVAL = 4, 1
else:
    BATCH_SIZE, GRAD_ACCUM_STEPS = 4, 4
    MAX_STEPS, WARMUP_STEPS = 34_000, 1_100       # warmup ≈ 3% of steps
    EVAL_INTERVAL, SAVE_INTERVAL = 500, 1000
    VAL_BATCHES, LOG_INTERVAL = 50, 25         # 50 x 4 = 200 val windows

EFFECTIVE_BATCH_SIZE = BATCH_SIZE * GRAD_ACCUM_STEPS

LEARNING_RATE = 3e-4
MIN_LR = 3e-5
WEIGHT_DECAY = 0.1
BETAS = (0.9, 0.95)
EPSILON = 1e-8
GRAD_CLIP = 1.0

# Gradient checkpointing saves memory but costs ~30% speed. Off by default.
USE_GRADIENT_CHECKPOINTING = False

# Stop + save before Kaggle's 12 h session limit (hours of training loop).
TIME_LIMIT_HOURS = 11.0

# Set True to continue from LAST_CHECKPOINT (e.g. after a Kaggle timeout).
RESUME_TRAINING = False


# ------------------------------------------------------------
# 6. REPRODUCIBILITY
# ------------------------------------------------------------

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


# ------------------------------------------------------------
# 7. DEVICE + PRECISION
#    bf16 only on Ampere+ (A100/L4/...). T4 (sm75) and P100 (sm60)
#    get fp16 + GradScaler: T4 reports bf16 "supported" but emulates it.
# ------------------------------------------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
    GPU_NAME = torch.cuda.get_device_name(0)
    CAPABILITY = torch.cuda.get_device_capability(0)
    AMP_DTYPE = torch.bfloat16 if CAPABILITY[0] >= 8 else torch.float16
    USE_AMP = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    GPU_NAME, CAPABILITY = "cpu", None
    AMP_DTYPE = torch.float32
    USE_AMP = False

USE_SCALER = AMP_DTYPE == torch.float16


def autocast_context():
    return torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP)


# ------------------------------------------------------------
# 8. GENERATION
# ------------------------------------------------------------

GEN_TEMPERATURE = 0.8
GEN_TOP_K = 50
GEN_TOP_P = 0.95
GEN_TARGET_WORDS = 128


# ------------------------------------------------------------
# 9. DISPLAY
# ------------------------------------------------------------

print("=" * 60)
print("ARCHITECTURE 2 — ENCODER-DECODER TRANSFORMER")
print("=" * 60)
print("DEBUG_MODE      :", DEBUG_MODE)
print("DEVICE          :", DEVICE, "|", GPU_NAME, "| capability", CAPABILITY)
print("AMP dtype       :", AMP_DTYPE, "| GradScaler:", USE_SCALER)
print("SRC_LEN/TGT_LEN :", SRC_LEN, "/", TGT_LEN)
print("Model           :", f"d={D_MODEL} heads={N_HEADS} layers={N_LAYERS}+{N_LAYERS} ffn={FFN_DIM}")
print("Batch           :", f"{BATCH_SIZE} x accum {GRAD_ACCUM_STEPS} = {EFFECTIVE_BATCH_SIZE}")
print("Steps           :", MAX_STEPS, "| warmup", WARMUP_STEPS)
print("Target tokens   :", f"{MAX_STEPS * EFFECTIVE_BATCH_SIZE * TGT_LEN:,}")
print("Grad checkpoint :", USE_GRADIENT_CHECKPOINTING)
print("=" * 60)

ARCHITECTURE 2 — ENCODER-DECODER TRANSFORMER
DEBUG_MODE      : False
DEVICE          : cuda | Tesla T4 | capability (7, 5)
AMP dtype       : torch.float16 | GradScaler: True
SRC_LEN/TGT_LEN : 1024 / 512
Model           : d=768 heads=12 layers=6+6 ffn=3072
Batch           : 4 x accum 4 = 16
Steps           : 34000 | warmup 1100
Target tokens   : 278,528,000
Grad checkpoint : False


### CELL 2 · Load the tokenizer

**In simple words:** Load our BPE tokenizer (`tokenizer.py` from the tokenizer notebook).

In [2]:
# ============================================================
# CELL 2 — LOAD THE SAVED V3 BPE TOKENIZER
# ============================================================
# We import tokenizer.py from the tokenizer notebook's output.
# It reads vocab.json + JSON-lines merges.txt correctly and
# its encode/decode round-trip exactly.

def find_input_dir(preferred, marker_file):
    """Return `preferred` if it holds marker_file, else search /kaggle/input."""
    preferred = Path(preferred)
    if (preferred / marker_file).exists():
        return preferred
    print(f"{marker_file} not found in {preferred}, searching /kaggle/input ...")
    hits = sorted(Path("/kaggle/input").rglob(marker_file))
    if not hits:
        raise FileNotFoundError(
            f"Could not find {marker_file}. Add the notebook output as an input."
        )
    print("Found:", hits[0].parent)
    return hits[0].parent


TOKENIZER_DIR = find_input_dir(TOKENIZER_DIR, "tokenizer_config.json")

for name in ["vocab.json", "merges.txt", "tokenizer_config.json", "tokenizer.py"]:
    assert (TOKENIZER_DIR / name).exists(), f"Missing {TOKENIZER_DIR / name}"


# ------------------------------------------------------------
# Import tokenizer.py under a unique module name
# ------------------------------------------------------------

spec = importlib.util.spec_from_file_location(
    "legal_bpe_tokenizer_v3", TOKENIZER_DIR / "tokenizer.py"
)
tokenizer_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tokenizer_module)

tokenizer, tokenizer_config = tokenizer_module.load_tokenizer(TOKENIZER_DIR)

PAD_ID = tokenizer.pad_id
UNK_ID = tokenizer.unk_id
BOS_ID = tokenizer.bos_id
EOS_ID = tokenizer.eos_id

SPECIAL_IDS = {PAD_ID, UNK_ID, BOS_ID, EOS_ID}


# ------------------------------------------------------------
# Checks
# ------------------------------------------------------------

assert tokenizer.get_vocab_size() == VOCAB_SIZE, tokenizer.get_vocab_size()
assert (PAD_ID, UNK_ID, BOS_ID, EOS_ID) == (0, 1, 2, 3), (PAD_ID, UNK_ID, BOS_ID, EOS_ID)

test_text = "The Supreme Court of India considered the matter.\n\nSection 12 applies."
test_ids = tokenizer.encode(test_text)
assert tokenizer.decode(test_ids) == tokenizer_module.normalize_text(test_text)
assert all(0 <= i < VOCAB_SIZE for i in test_ids)


# ------------------------------------------------------------
# Keep a copy of the tokenizer next to the checkpoints
# ------------------------------------------------------------

SAVED_TOKENIZER_DIR = OUTPUT_DIR / "tokenizer"
SAVED_TOKENIZER_DIR.mkdir(exist_ok=True)
for name in ["vocab.json", "merges.txt", "tokenizer_config.json", "tokenizer.py"]:
    shutil.copy2(TOKENIZER_DIR / name, SAVED_TOKENIZER_DIR / name)


print("Tokenizer dir :", TOKENIZER_DIR)
print("Vocab size    :", tokenizer.get_vocab_size())
print("Merges        :", len(tokenizer.merges))
print("PAD/UNK/BOS/EOS:", PAD_ID, UNK_ID, BOS_ID, EOS_ID)
print("Test tokens   :", len(test_ids), test_ids[:12])
print("\n✅ Tokenizer loaded and round-trip verified.")

Tokenizer dir : /kaggle/input/notebooks/ritikguptatab/updated-bpe-tokenizer/legal_tokenizer_v3/tokenizer/final_tokenizer
Vocab size    : 32004
Merges        : 31438
PAD/UNK/BOS/EOS: 0 1 2 3
Test tokens   : 24 [1052, 12, 31296, 12, 1488, 12, 1382, 12, 9331, 12, 28388, 12]

✅ Tokenizer loaded and round-trip verified.


### CELL 3 · Load the tokenized corpus into RAM

**In simple words:** Load the four token files into RAM (≈ 1.8 GB) so random windows are read instantly.

**How it works:** The metadata of the tokenization notebook is checked first.

In [3]:
# ============================================================
# CELL 3 — LOAD TOKENIZED CORPUS (uint16) INTO RAM
# ============================================================
# ~1.8 GB total. Loading into RAM (Kaggle has ~29 GB) makes the
# random window reads ~free compared to reading /kaggle/input.

TOKEN_DIR = find_input_dir(TOKEN_DIR, TOKEN_FILE_PATTERN.format(source=SOURCES[0]))

# ------------------------------------------------------------
# Check metadata written by the tokenization notebook
# ------------------------------------------------------------

metadata_path = TOKEN_DIR / "tokenization_metadata.json"

if metadata_path.exists():
    with open(metadata_path, "r", encoding="utf-8") as f:
        token_metadata = json.load(f)
    print("Metadata:", token_metadata["tokenizer"])
    assert token_metadata["tokenizer"]["vocab_size"] == VOCAB_SIZE
    assert token_metadata["tokenizer"]["eos_id"] == EOS_ID
    assert token_metadata["tokenizer"]["dtype"] == "uint16"
else:
    token_metadata = None
    print("No tokenization_metadata.json found — skipping metadata check.")


# ------------------------------------------------------------
# Load every source
# ------------------------------------------------------------

token_arrays = {}

for source in SOURCES:

    path = TOKEN_DIR / TOKEN_FILE_PATTERN.format(source=source)
    assert path.exists(), f"Missing {path}"

    start = time.time()
    tokens = np.fromfile(path, dtype=TOKEN_DTYPE)

    max_id = int(tokens.max())
    eos_count = int(np.count_nonzero(tokens == EOS_ID))

    assert max_id < VOCAB_SIZE, f"{source}: token id {max_id} >= vocab {VOCAB_SIZE}"
    assert eos_count > 0, f"{source}: no EOS tokens — wrong file?"

    if token_metadata is not None:
        assert len(tokens) == token_metadata["sources"][source]["tokens"], source

    token_arrays[source] = tokens

    print(
        f"{source:15s} tokens={len(tokens):>13,} | docs(EOS)={eos_count:>8,} | "
        f"max_id={max_id} | {path.stat().st_size / 1024**3:.2f} GB | {time.time() - start:.1f}s"
    )

total_tokens = sum(len(t) for t in token_arrays.values())
print(f"\nTotal tokens: {total_tokens:,}")

# Decode a short slice to see real text
print("\nSample decoded text (acts):")
print(repr(tokenizer.decode(token_arrays["acts"][:60].tolist())))

print("\n✅ Tokenized corpus loaded.")

Metadata: {'type': 'custom_bpe', 'vocab_size': 32004, 'eos_token': '<eos>', 'eos_id': 3, 'dtype': 'uint16'}
acts            tokens=  256,521,619 | docs(EOS)=  15,490 | max_id=32003 | 0.48 GB | 19.3s
high_courts     tokens=  242,356,017 | docs(EOS)= 223,628 | max_id=32003 | 0.45 GB | 21.7s
supreme_court   tokens=  241,501,699 | docs(EOS)=  26,627 | max_id=32003 | 0.45 GB | 13.6s
tribunals       tokens=  241,779,387 | docs(EOS)=  57,691 | max_id=32003 | 0.45 GB | 20.8s

Total tokens: 982,158,722

Sample decoded text (acts):

✅ Tokenized corpus loaded.


### CELL 4 · Train / validation split

**In simple words:** Per source: training tokens come from the start (up to a budget), validation is the last 1% of the file — the two never overlap.

In [4]:
# ============================================================
# CELL 4 — TRAIN / VALIDATION SPLIT PER SOURCE
# ============================================================
# Per source: train = first TRAIN_TOKEN_BUDGET[source] tokens,
#             val   = last 1% of the file.

split_ranges = {"train": {}, "val": {}}
train_tokens_used = {}

for source in SOURCES:

    n = len(token_arrays[source])
    val_start = int(n * (1.0 - VAL_FRACTION))

    # Train never reaches into the validation region
    train_end = min(TRAIN_TOKEN_BUDGET[source], val_start)
    train_tokens_used[source] = train_end

    if train_end < TRAIN_TOKEN_BUDGET[source]:
        print(f"⚠ {source}: only {train_end:,} train tokens available "
              f"(budget {TRAIN_TOKEN_BUDGET[source]:,})")

    regions = {
        "train": (0, train_end),
        "val": (val_start, n),
    }

    for split, (region_start, region_end) in regions.items():
        max_start = region_end - WINDOW_LEN
        assert max_start >= region_start, f"{source}/{split} too small for {WINDOW_LEN}"
        split_ranges[split][source] = (region_start, max_start)

    print(
        f"{source:15s} train={train_end:>13,} tokens "
        f"({MIXING_RATIOS[source]:.0%}) | val={n - val_start:>11,} tokens"
    )


# Sampling uses the same ratios, so batches follow the mix too
source_probabilities = np.array([MIXING_RATIOS[s] for s in SOURCES], dtype=np.float64)
assert np.isclose(source_probabilities.sum(), 1.0)

total_train_pool = sum(train_tokens_used.values())
tokens_processed = MAX_STEPS * EFFECTIVE_BATCH_SIZE * WINDOW_LEN

print(f"\nTrain pool       : {total_train_pool:,} tokens (limit {MAX_TRAIN_TOKENS:,})")
print(f"Tokens processed : {tokens_processed:,} "
      f"(= {tokens_processed / total_train_pool:.2f} x pool in {MAX_STEPS} steps)")
print("Mixing           :", {s: float(p) for s, p in zip(SOURCES, source_probabilities)})
print("\n✅ Splits ready.")

acts            train=  120,000,000 tokens (15%) | val=  5,130,433 tokens
⚠ high_courts: only 237,508,896 train tokens available (budget 280,000,000)
high_courts     train=  237,508,896 tokens (35%) | val=  4,847,121 tokens
supreme_court   train=  200,000,000 tokens (25%) | val=  4,830,034 tokens
tribunals       train=  200,000,000 tokens (25%) | val=  4,835,588 tokens

Train pool       : 757,508,896 tokens (limit 800,000,000)
Tokens processed : 835,584,000 (= 1.10 x pool in 34000 steps)
Mixing           : {'acts': 0.15, 'high_courts': 0.35, 'supreme_court': 0.25, 'tribunals': 0.25}

✅ Splits ready.


### CELL 5 · Batch sampler

**In simple words:** One example = one random window: the encoder reads its first 1,024 tokens, the decoder learns to write the next 512.

**How it works:** Output removed before publishing.

In [5]:
# ============================================================
# CELL 5 — BATCH SAMPLER (no DataLoader needed)
# ============================================================
# Each example = one random window of WINDOW_LEN tokens:
#
#   window          : t0 ........ t(S-1) | y0 y1 ... y(T-1)
#   encoder_input   : t0 ... t(S-1)
#   decoder_input   : <bos> y0 ... y(T-2)
#   decoder_target  :       y0 ... y(T-1)
#
# Sampling 4 windows from RAM takes microseconds, so a plain
# function is faster and simpler than DataLoader workers.

def sample_batch(split, batch_size, rng):

    chosen = rng.choice(len(SOURCES), size=batch_size, p=source_probabilities)

    encoder_input = np.empty((batch_size, SRC_LEN), dtype=np.int64)
    decoder_input = np.empty((batch_size, TGT_LEN), dtype=np.int64)
    decoder_target = np.empty((batch_size, TGT_LEN), dtype=np.int64)

    for b, source_index in enumerate(chosen):

        source = SOURCES[source_index]
        low, high = split_ranges[split][source]
        start = int(rng.integers(low, high + 1))

        window = token_arrays[source][start:start + WINDOW_LEN]

        encoder_input[b] = window[:SRC_LEN]
        decoder_target[b] = window[SRC_LEN:]
        decoder_input[b, 0] = BOS_ID
        decoder_input[b, 1:] = window[SRC_LEN:-1]

    return (
        torch.from_numpy(encoder_input),
        torch.from_numpy(decoder_input),
        torch.from_numpy(decoder_target),
    )


def to_device(batch):
    if DEVICE.type == "cuda":
        return tuple(x.pin_memory().to(DEVICE, non_blocking=True) for x in batch)
    return tuple(x.to(DEVICE) for x in batch)


# ------------------------------------------------------------
# Fixed validation batches -> val loss is comparable across steps
# ------------------------------------------------------------

val_rng = np.random.default_rng(SEED + 10_000)
val_batches = [sample_batch("val", BATCH_SIZE, val_rng) for _ in range(VAL_BATCHES)]


# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

check_rng = np.random.default_rng(SEED)
enc, dec, tgt = sample_batch("train", BATCH_SIZE, check_rng)

assert enc.shape == (BATCH_SIZE, SRC_LEN)
assert dec.shape == tgt.shape == (BATCH_SIZE, TGT_LEN)
assert torch.all(dec[:, 0] == BOS_ID)
assert torch.equal(dec[:, 1:], tgt[:, :-1])            # teacher-forcing shift
assert int(enc.max()) < VOCAB_SIZE and int(tgt.max()) < VOCAB_SIZE

print("Encoder input :", tuple(enc.shape))
print("Decoder input :", tuple(dec.shape), dec[0, :8].tolist())
print("Target        :", tuple(tgt.shape), tgt[0, :8].tolist())
print("\nEncoder text (end) :", repr(tokenizer.decode(enc[0, -30:].tolist())))
print("Target text (start):", repr(tokenizer.decode(tgt[0, :30].tolist())))
print("\nValidation batches:", len(val_batches))
print("\n✅ Batch sampler verified.")

[output removed before publishing: this cell printed raw dataset / test text]


### CELL 6 · Multi-head attention

**In simple words:** The attention used everywhere: self-attention, and cross-attention (the decoder looking at the encoder).

In [6]:
# ============================================================
# CELL 6 — MULTI-HEAD ATTENTION (unchanged)
# ============================================================

class MultiHeadAttention(nn.Module):

    def __init__(self, d_model, n_heads, dropout=0.0, bias=True):
        super().__init__()

        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"

        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.dropout = dropout

        self.q_proj = nn.Linear(d_model, d_model, bias=bias)
        self.k_proj = nn.Linear(d_model, d_model, bias=bias)
        self.v_proj = nn.Linear(d_model, d_model, bias=bias)
        self.out_proj = nn.Linear(d_model, d_model, bias=bias)

    def _reshape_heads(self, x):
        # [B, T, D] -> [B, H, T, head_dim]
        B, T, _ = x.shape
        return x.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

    def forward(self, query, key=None, value=None, causal=False):

        # Self-attention when key/value are not given
        if key is None:
            key = query
        if value is None:
            value = key

        q = self._reshape_heads(self.q_proj(query))
        k = self._reshape_heads(self.k_proj(key))
        v = self._reshape_heads(self.v_proj(value))

        # Fused attention kernel (flash / memory-efficient when available)
        out = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=None,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=causal,
        )

        # [B, H, T, head_dim] -> [B, T, D]
        out = out.transpose(1, 2).contiguous().view(query.size(0), query.size(1), self.d_model)

        return self.out_proj(out)


# Quick shape check
_attn = MultiHeadAttention(D_MODEL, N_HEADS)
_x = torch.randn(2, 16, D_MODEL)
_m = torch.randn(2, 32, D_MODEL)
assert _attn(_x, causal=True).shape == (2, 16, D_MODEL)
assert _attn(_x, key=_m, value=_m).shape == (2, 16, D_MODEL)
del _attn, _x, _m

print("✅ MultiHeadAttention ready.")

✅ MultiHeadAttention ready.


### CELL 7 · Positions + feed-forward

**In simple words:** Learned position embeddings (where a token is) and the feed-forward layer (GELU, 3,072).

In [7]:
# ============================================================
# CELL 7 — POSITIONAL EMBEDDING + FEED-FORWARD (unchanged)
# ============================================================

class PositionalEmbedding(nn.Module):
    """Learned absolute positions added to token embeddings."""

    def __init__(self, max_len, d_model, dropout=0.0):
        super().__init__()
        self.max_len = max_len
        self.embedding = nn.Embedding(max_len, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        T = x.size(1)
        if T > self.max_len:
            raise ValueError(f"Sequence length {T} > max_len {self.max_len}")
        positions = torch.arange(T, device=x.device)
        return self.dropout(x + self.embedding(positions).unsqueeze(0))


class FeedForward(nn.Module):

    def __init__(self, d_model, d_ff, dropout=0.0, bias=True):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff, bias=bias)
        self.fc2 = nn.Linear(d_ff, d_model, bias=bias)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.fc2(F.gelu(self.fc1(x))))


print("✅ PositionalEmbedding + FeedForward ready.")

✅ PositionalEmbedding + FeedForward ready.


### CELL 8 · Encoder block + decoder block

**In simple words:** Encoder block: attention over all positions, both directions. Decoder block: masked self-attention (no peeking ahead) + cross-attention to the encoder.

**How it works:** Pre-LayerNorm: the norm comes before every sub-layer.

In [8]:
# ============================================================
# CELL 8 — ENCODER BLOCK + DECODER BLOCK (Pre-LN, unchanged)
# ============================================================

class EncoderBlock(nn.Module):

    def __init__(self, d_model, n_heads, d_ff, dropout=0.0, bias=True):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.self_attention = MultiHeadAttention(d_model, n_heads, dropout, bias)
        self.norm2 = nn.LayerNorm(d_model)
        self.feed_forward = FeedForward(d_model, d_ff, dropout, bias)

    def forward(self, x):
        # Encoder self-attention is NOT causal
        x = x + self.self_attention(self.norm1(x), causal=False)
        x = x + self.feed_forward(self.norm2(x))
        return x


class DecoderBlock(nn.Module):

    def __init__(self, d_model, n_heads, d_ff, dropout=0.0, bias=True):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.self_attention = MultiHeadAttention(d_model, n_heads, dropout, bias)
        self.norm2 = nn.LayerNorm(d_model)
        self.cross_attention = MultiHeadAttention(d_model, n_heads, dropout, bias)
        self.norm3 = nn.LayerNorm(d_model)
        self.feed_forward = FeedForward(d_model, d_ff, dropout, bias)

    def forward(self, x, encoder_output):
        # 1. masked self-attention (no peeking at future targets)
        x = x + self.self_attention(self.norm1(x), causal=True)
        # 2. cross-attention: Q = decoder, K/V = encoder
        x = x + self.cross_attention(
            self.norm2(x), key=encoder_output, value=encoder_output, causal=False
        )
        # 3. feed-forward
        x = x + self.feed_forward(self.norm3(x))
        return x


print("✅ EncoderBlock + DecoderBlock ready.")

✅ EncoderBlock + DecoderBlock ready.


### CELL 9 · Full encoder + full decoder

**In simple words:** Stacks of 6 encoder blocks and 6 decoder blocks.

In [9]:
# ============================================================
# CELL 9 — FULL ENCODER + FULL DECODER (unchanged)
# ============================================================

class TransformerEncoder(nn.Module):

    def __init__(self, vocab_size, max_len, d_model, n_heads, n_layers, d_ff,
                 dropout=0.0, bias=True):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = PositionalEmbedding(max_len, d_model, dropout)
        self.layers = nn.ModuleList([
            EncoderBlock(d_model, n_heads, d_ff, dropout, bias) for _ in range(n_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)

    def forward(self, input_ids):
        x = self.position_embedding(self.token_embedding(input_ids))
        for layer in self.layers:
            if USE_GRADIENT_CHECKPOINTING and self.training:
                x = torch.utils.checkpoint.checkpoint(layer, x, use_reentrant=False)
            else:
                x = layer(x)
        return self.final_norm(x)


class TransformerDecoder(nn.Module):

    def __init__(self, vocab_size, max_len, d_model, n_heads, n_layers, d_ff,
                 dropout=0.0, bias=True):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = PositionalEmbedding(max_len, d_model, dropout)
        self.layers = nn.ModuleList([
            DecoderBlock(d_model, n_heads, d_ff, dropout, bias) for _ in range(n_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)

    def forward(self, input_ids, encoder_output):
        x = self.position_embedding(self.token_embedding(input_ids))
        for layer in self.layers:
            if USE_GRADIENT_CHECKPOINTING and self.training:
                x = torch.utils.checkpoint.checkpoint(
                    layer, x, encoder_output, use_reentrant=False
                )
            else:
                x = layer(x, encoder_output)
        return self.final_norm(x)


print("✅ TransformerEncoder + TransformerDecoder ready.")

✅ TransformerEncoder + TransformerDecoder ready.


### CELL 10 · Encoder-decoder LM + initialization

**In simple words:** The whole model (149.6M parameters). The first loss ≈ 10.27 is close to the expected ln(32,004) ≈ 10.37 of an untrained model — a sign the setup is right.

**How it works:** Decoder embedding tied with the output layer; the loss is computed in float32.

In [10]:
# ============================================================
# CELL 10 — ENCODER-DECODER LM + WEIGHT INIT
# ============================================================

class EncoderDecoderLM(nn.Module):

    def __init__(self, vocab_size, src_max_len, tgt_max_len, d_model, n_heads,
                 encoder_layers, decoder_layers, d_ff, dropout=0.0, bias=True):
        super().__init__()

        self.encoder = TransformerEncoder(
            vocab_size, src_max_len, d_model, n_heads, encoder_layers, d_ff, dropout, bias
        )
        self.decoder = TransformerDecoder(
            vocab_size, tgt_max_len, d_model, n_heads, decoder_layers, d_ff, dropout, bias
        )

        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        # Weight tying: decoder embedding <-> output projection
        self.lm_head.weight = self.decoder.token_embedding.weight

    def forward(self, encoder_input_ids, decoder_input_ids, targets=None):

        encoder_output = self.encoder(encoder_input_ids)
        decoder_output = self.decoder(decoder_input_ids, encoder_output)
        logits = self.lm_head(decoder_output)

        loss = None
        if targets is not None:
            # Loss in fp32 for stability under fp16 autocast
            loss = F.cross_entropy(
                logits.float().view(-1, logits.size(-1)),
                targets.reshape(-1),
                ignore_index=PAD_ID,
            )

        return logits, loss


def initialize_transformer_weights(module):
    if isinstance(module, nn.Linear):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if module.bias is not None:
            nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)
    elif isinstance(module, nn.LayerNorm):
        nn.init.ones_(module.weight)
        nn.init.zeros_(module.bias)


# ------------------------------------------------------------
# Build model
# ------------------------------------------------------------

seq2seq_model = EncoderDecoderLM(
    vocab_size=VOCAB_SIZE,
    src_max_len=SRC_LEN,
    tgt_max_len=TGT_LEN,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    encoder_layers=ENCODER_LAYERS,
    decoder_layers=DECODER_LAYERS,
    d_ff=FFN_DIM,
    dropout=DROPOUT,
)

seq2seq_model.apply(initialize_transformer_weights)
seq2seq_model.lm_head.weight = seq2seq_model.decoder.token_embedding.weight
seq2seq_model = seq2seq_model.to(DEVICE)

assert seq2seq_model.lm_head.weight is seq2seq_model.decoder.token_embedding.weight

total_params = sum(p.numel() for p in seq2seq_model.parameters())
print(f"Total parameters: {total_params:,}")


# ------------------------------------------------------------
# Initial loss should be ~ ln(32004) = 10.37
# ------------------------------------------------------------

seq2seq_model.eval()
with torch.no_grad(), autocast_context():
    _, init_loss = seq2seq_model(*to_device(val_batches[0]))
print(f"Initial loss: {init_loss.item():.4f} (expected ~{math.log(VOCAB_SIZE):.2f})")
assert torch.isfinite(init_loss) and init_loss.item() < 12.0

print("\n✅ Model built.")

Total parameters: 149,578,752
Initial loss: 10.2660 (expected ~10.37)

✅ Model built.


### CELL 11 · Optimizer, schedule, grad scaler

**In simple words:** AdamW (weight decay on matrices only), warm-up then cosine decay, GradScaler for float16.

In [11]:
# ============================================================
# CELL 11 — OPTIMIZER + LR SCHEDULE + GRAD SCALER
# ============================================================

# Weight decay on matrices; no decay on biases / LayerNorm
decay_params = [p for p in seq2seq_model.parameters() if p.requires_grad and p.dim() >= 2]
no_decay_params = [p for p in seq2seq_model.parameters() if p.requires_grad and p.dim() < 2]

assert sum(p.numel() for p in decay_params) + sum(p.numel() for p in no_decay_params) == total_params

optimizer = torch.optim.AdamW(
    [
        {"params": decay_params, "weight_decay": WEIGHT_DECAY},
        {"params": no_decay_params, "weight_decay": 0.0},
    ],
    lr=LEARNING_RATE,
    betas=BETAS,
    eps=EPSILON,
    fused=DEVICE.type == "cuda",     # one fused kernel for the update
)

scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)


def get_learning_rate(step):
    """Linear warmup, then cosine decay to MIN_LR."""
    if step < WARMUP_STEPS:
        return LEARNING_RATE * (step + 1) / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / max(1, MAX_STEPS - WARMUP_STEPS)
    progress = min(1.0, max(0.0, progress))
    return MIN_LR + (LEARNING_RATE - MIN_LR) * 0.5 * (1.0 + math.cos(math.pi * progress))


for s in [0, WARMUP_STEPS - 1, WARMUP_STEPS, MAX_STEPS // 2, MAX_STEPS]:
    print(f"step={s:5d} lr={get_learning_rate(s):.3e}")

print("\n✅ Optimizer ready (fused AdamW:", DEVICE.type == "cuda", "| scaler:", USE_SCALER, ")")

step=    0 lr=2.727e-07
step= 1099 lr=3.000e-04
step= 1100 lr=3.000e-04
step=17000 lr=1.721e-04
step=34000 lr=3.000e-05

✅ Optimizer ready (fused AdamW: True | scaler: True )


### CELL 12 · Validation + checkpoints

**In simple words:** Functions for the validation loss and for saving / loading checkpoints.

In [12]:
# ============================================================
# CELL 12 — VALIDATION + CHECKPOINT UTILITIES
# ============================================================

@torch.no_grad()
def evaluate_model(model):
    """Mean loss over the fixed validation batches."""
    model.eval()
    total = torch.zeros((), device=DEVICE)
    for batch in val_batches:
        with autocast_context():
            _, loss = model(*to_device(batch))
        total += loss.float()
    model.train()
    val_loss = (total / len(val_batches)).item()       # one sync
    return val_loss, math.exp(min(val_loss, 20.0))


MODEL_CONFIG = {
    "architecture": "encoder_decoder",
    "vocab_size": VOCAB_SIZE,
    "src_len": SRC_LEN,
    "tgt_len": TGT_LEN,
    "d_model": D_MODEL,
    "n_heads": N_HEADS,
    "encoder_layers": ENCODER_LAYERS,
    "decoder_layers": DECODER_LAYERS,
    "ffn_dim": FFN_DIM,
    "dropout": DROPOUT,
    "pad_id": PAD_ID,
    "unk_id": UNK_ID,
    "bos_id": BOS_ID,
    "eos_id": EOS_ID,
    "tokenizer_dir": str(SAVED_TOKENIZER_DIR),
}


def save_checkpoint(path, step, best_val_loss, full_state=True):
    """full_state=True -> also optimizer/scaler/RNG (for resume)."""
    checkpoint = {
        "step": int(step),
        "best_val_loss": float(best_val_loss),
        "model_state_dict": seq2seq_model.state_dict(),
        "config": MODEL_CONFIG,
    }
    if full_state:
        checkpoint["optimizer_state_dict"] = optimizer.state_dict()
        checkpoint["scaler_state_dict"] = scaler.state_dict()

    tmp = Path(str(path) + ".tmp")
    torch.save(checkpoint, tmp)
    os.replace(tmp, path)                  # never leaves a half-written file


def load_checkpoint(path, load_optimizer=True):
    checkpoint = torch.load(path, map_location=DEVICE, weights_only=False)
    seq2seq_model.load_state_dict(checkpoint["model_state_dict"])
    if load_optimizer and "optimizer_state_dict" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        scaler.load_state_dict(checkpoint["scaler_state_dict"])
    return checkpoint["step"], checkpoint["best_val_loss"]


with open(OUTPUT_DIR / "model_config.json", "w") as f:
    json.dump(MODEL_CONFIG, f, indent=2)

print("✅ Validation + checkpoint utilities ready.")

✅ Validation + checkpoint utilities ready.


### CELL 13 · One-step memory / speed check

**In simple words:** One forward + backward pass before training, so an out-of-memory error shows up now, not hours later; also estimates the total training time.

In [13]:
# ============================================================
# CELL 13 — ONE-STEP MEMORY / SPEED CHECK
# ============================================================
# Runs forward + backward on one micro-batch (no optimizer step)
# so an OOM shows up here, not hours into training.

seq2seq_model.train()
optimizer.zero_grad(set_to_none=True)

if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

probe_rng = np.random.default_rng(SEED + 1)
timings = []

for i in range(3):
    batch = to_device(sample_batch("train", BATCH_SIZE, probe_rng))
    t0 = time.time()
    with autocast_context():
        _, loss = seq2seq_model(*batch)
    scaler.scale(loss / GRAD_ACCUM_STEPS).backward()
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    timings.append(time.time() - t0)

optimizer.zero_grad(set_to_none=True)

micro_time = min(timings[1:])          # first call includes warm-up
print(f"Loss            : {loss.item():.4f}")
print(f"Micro-batch time: {micro_time:.3f} s")
print(f"Est. step time  : {micro_time * GRAD_ACCUM_STEPS:.2f} s")
print(f"Est. total time : {micro_time * GRAD_ACCUM_STEPS * MAX_STEPS / 3600:.2f} h "
      f"(+ validation/checkpoints)")

if DEVICE.type == "cuda":
    peak = torch.cuda.max_memory_allocated() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"Peak GPU memory : {peak:.2f} / {total:.1f} GB")

del batch, loss
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

print("\n✅ Memory/speed check passed.")

Loss            : 10.2038
Micro-batch time: 0.214 s
Est. step time  : 0.86 s
Est. total time : 8.10 h (+ validation/checkpoints)
Peak GPU memory : 3.63 / 14.6 GB

✅ Memory/speed check passed.


### CELL 14 · Training loop

**In simple words:** The training (34,000 updates): random windows, gradient accumulation, validation and checkpoints; it stops safely before Kaggle's time limit.

In [14]:
# ============================================================
# CELL 14 — TRAINING LOOP
# ============================================================

print("=" * 70)
print("STARTING ENCODER-DECODER TRAINING")
print("=" * 70)

global_step = 0
best_val_loss = float("inf")

if RESUME_TRAINING and LAST_CHECKPOINT.exists():
    global_step, best_val_loss = load_checkpoint(LAST_CHECKPOINT)
    print(f"Resumed from step {global_step} | best val loss {best_val_loss:.4f}")
else:
    print("Starting from scratch.")

# Different random stream after a resume -> no repeated batches
train_rng = np.random.default_rng(SEED + global_step)

seq2seq_model.train()

history = []
training_start = time.time()
log_start = time.time()
log_loss = torch.zeros((), device=DEVICE)
log_steps = 0
stopped_by_time = False

while global_step < MAX_STEPS:

    # --------------------------------------------------------
    # Learning rate for this step
    # --------------------------------------------------------
    lr = get_learning_rate(global_step)
    for group in optimizer.param_groups:
        group["lr"] = lr

    optimizer.zero_grad(set_to_none=True)
    step_loss = torch.zeros((), device=DEVICE)

    # --------------------------------------------------------
    # Gradient accumulation
    # --------------------------------------------------------
    for micro_step in range(GRAD_ACCUM_STEPS):

        encoder_input, decoder_input, targets = to_device(
            sample_batch("train", BATCH_SIZE, train_rng)
        )

        with autocast_context():
            _, loss = seq2seq_model(encoder_input, decoder_input, targets)

        scaler.scale(loss / GRAD_ACCUM_STEPS).backward()
        step_loss += loss.detach() / GRAD_ACCUM_STEPS

    # --------------------------------------------------------
    # Clip + optimizer step
    # --------------------------------------------------------
    scaler.unscale_(optimizer)
    grad_norm = torch.nn.utils.clip_grad_norm_(seq2seq_model.parameters(), GRAD_CLIP)

    scaler.step(optimizer)       # skipped automatically if grads are inf/nan (fp16)
    scaler.update()

    global_step += 1
    log_loss += step_loss
    log_steps += 1

    # --------------------------------------------------------
    # Logging (only here do we sync with the GPU)
    # --------------------------------------------------------
    if global_step == 1 or global_step % LOG_INTERVAL == 0:

        avg_loss = (log_loss / log_steps).item()
        if not math.isfinite(avg_loss):
            raise RuntimeError(f"Non-finite loss at step {global_step}: {avg_loss}")

        elapsed = time.time() - training_start
        sec_per_step = (time.time() - log_start) / log_steps
        eta_h = sec_per_step * (MAX_STEPS - global_step) / 3600

        print(
            f"Step {global_step:5d}/{MAX_STEPS} | loss={avg_loss:.4f} | lr={lr:.3e} | "
            f"grad_norm={float(grad_norm):.3f} | {sec_per_step:.2f}s/step | "
            f"elapsed={elapsed / 60:.1f}m | eta={eta_h:.2f}h"
        )

        history.append({"step": global_step, "train_loss": avg_loss, "lr": lr})
        log_loss.zero_()
        log_steps = 0
        log_start = time.time()

    # --------------------------------------------------------
    # Validation + best checkpoint
    # --------------------------------------------------------
    if global_step % EVAL_INTERVAL == 0 or global_step == MAX_STEPS:

        val_loss, val_ppl = evaluate_model(seq2seq_model)
        history.append({"step": global_step, "val_loss": val_loss})

        message = f"\n[Validation] step {global_step} | val_loss={val_loss:.4f} | ppl={val_ppl:.2f}"

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_checkpoint(BEST_CHECKPOINT, global_step, best_val_loss, full_state=False)
            message += " | ⭐ new best saved"

        print(message + "\n")

    # --------------------------------------------------------
    # Resume checkpoint
    # --------------------------------------------------------
    if global_step % SAVE_INTERVAL == 0 or global_step == MAX_STEPS:
        save_checkpoint(LAST_CHECKPOINT, global_step, best_val_loss, full_state=True)

    # --------------------------------------------------------
    # Kaggle time limit
    # --------------------------------------------------------
    if time.time() - training_start > TIME_LIMIT_HOURS * 3600:
        save_checkpoint(LAST_CHECKPOINT, global_step, best_val_loss, full_state=True)
        stopped_by_time = True
        print(f"\n⏱ Time limit reached at step {global_step}. Saved {LAST_CHECKPOINT}.")
        print("Set RESUME_TRAINING = True (and add this notebook's output as input) to continue.")
        break


with open(OUTPUT_DIR / "training_history.json", "w") as f:
    json.dump(history, f, indent=2)

print("=" * 70)
print("✅ TRAINING FINISHED" if not stopped_by_time else "⏱ TRAINING PAUSED")
print("=" * 70)
print("Final step          :", global_step)
print("Best validation loss:", best_val_loss)
print(f"Training time       : {(time.time() - training_start) / 3600:.2f} h")
print("Best checkpoint     :", BEST_CHECKPOINT)
print("Last checkpoint     :", LAST_CHECKPOINT)

STARTING ENCODER-DECODER TRAINING
Starting from scratch.
Step     1/34000 | loss=10.2035 | lr=2.727e-07 | grad_norm=39.335 | 1.30s/step | elapsed=0.0m | eta=12.29h
Step    25/34000 | loss=8.0516 | lr=6.818e-06 | grad_norm=7.522 | 0.91s/step | elapsed=0.4m | eta=8.57h
Step    50/34000 | loss=6.3982 | lr=1.364e-05 | grad_norm=1.968 | 0.95s/step | elapsed=0.8m | eta=8.95h
Step    75/34000 | loss=5.9162 | lr=2.045e-05 | grad_norm=2.703 | 1.01s/step | elapsed=1.2m | eta=9.52h
Step   100/34000 | loss=5.3753 | lr=2.727e-05 | grad_norm=2.158 | 1.08s/step | elapsed=1.7m | eta=10.21h
Step   125/34000 | loss=4.9831 | lr=3.409e-05 | grad_norm=1.987 | 1.03s/step | elapsed=2.1m | eta=9.69h
Step   150/34000 | loss=4.6618 | lr=4.091e-05 | grad_norm=3.288 | 1.02s/step | elapsed=2.5m | eta=9.60h
Step   175/34000 | loss=4.3669 | lr=4.773e-05 | grad_norm=1.908 | 1.03s/step | elapsed=2.9m | eta=9.72h
Step   200/34000 | loss=4.1703 | lr=5.455e-05 | grad_norm=0.957 | 1.03s/step | elapsed=3.4m | eta=9.70h
Ste

### CELL 15 · Load the best checkpoint + verify

**In simple words:** Load the best checkpoint and re-measure its validation loss (1.4996, perplexity 4.48).

In [15]:
# ============================================================
# CELL 15 — LOAD BEST CHECKPOINT + VERIFY
# ============================================================

checkpoint = torch.load(BEST_CHECKPOINT, map_location=DEVICE, weights_only=False)

for key in ["vocab_size", "src_len", "tgt_len", "d_model", "n_heads",
            "encoder_layers", "decoder_layers", "ffn_dim", "bos_id", "eos_id"]:
    assert checkpoint["config"][key] == MODEL_CONFIG[key], key

seq2seq_model.load_state_dict(checkpoint["model_state_dict"])

restored_val_loss, restored_val_ppl = evaluate_model(seq2seq_model)

print("Saved step          :", checkpoint["step"])
print("Saved best val loss :", round(checkpoint["best_val_loss"], 4))
print("Restored val loss   :", round(restored_val_loss, 4))
print("Restored perplexity :", round(restored_val_ppl, 2))

# Same fixed val batches + eval mode -> should match closely
assert abs(restored_val_loss - checkpoint["best_val_loss"]) < 1e-2

print("\n✅ Best checkpoint restored and verified.")

Saved step          : 34000
Saved best val loss : 1.4996
Restored val loss   : 1.4996
Restored perplexity : 4.48

✅ Best checkpoint restored and verified.


### CELL 16 · Generation

**In simple words:** Generation with top-k / top-p sampling and anti-repetition rules (a repetition penalty, no repeated 12-grams).

In [16]:
# ============================================================
# CELL 16 — GENERATION (top-k / top-p + anti-repetition)
# ============================================================

# ------------------------------------------------------------
# Anti-repetition settings (tune these)
# ------------------------------------------------------------

REPETITION_PENALTY = 1.15   # 1.0 = off. Lowers scores of recently used tokens
REPETITION_WINDOW = 64      # how many recent generated tokens get the penalty
NO_REPEAT_NGRAM = 12        # never repeat a 12-token phrase (~5 words). 0 = off
ALLOW_EARLY_EOS = False     # False -> always reach target_words


def apply_repetition_penalty(logits, generated, penalty, window):
    """Lower the score of tokens used in the last `window` tokens."""
    if penalty == 1.0:
        return logits

    recent = [t for t in generated[-window:] if t not in SPECIAL_IDS]
    if not recent:
        return logits

    ids = torch.tensor(sorted(set(recent)), device=logits.device)
    scores = logits[ids]

    # positive score -> divide, negative score -> multiply (both make it less likely)
    logits[ids] = torch.where(scores > 0, scores / penalty, scores * penalty)
    return logits


def banned_ngram_tokens(generated, n):
    """Tokens that would complete an n-gram already present in `generated`."""
    if n <= 0 or len(generated) < n:
        return []

    prefix = generated[-(n - 1):]          # last n-1 tokens
    banned = []

    for i in range(len(generated) - n + 1):
        if generated[i:i + n - 1] == prefix:
            banned.append(generated[i + n - 1])

    return banned


@torch.no_grad()
def sample_next_token(logits, generated, temperature=0.8, top_k=50, top_p=0.95):

    logits = logits.float().clone()

    # 1. Anti-repetition (before temperature)
    logits = apply_repetition_penalty(logits, generated, REPETITION_PENALTY, REPETITION_WINDOW)

    for token_id in banned_ngram_tokens(generated, NO_REPEAT_NGRAM):
        logits[token_id] = -float("inf")

    # 2. Never emit PAD / BOS / UNK (and EOS unless allowed)
    logits[PAD_ID] = -float("inf")
    logits[BOS_ID] = -float("inf")
    logits[UNK_ID] = -float("inf")
    if not ALLOW_EARLY_EOS:
        logits[EOS_ID] = -float("inf")

    # 3. Temperature
    logits = logits / temperature

    # 4. Top-k
    if top_k is not None and top_k > 0:
        k = min(top_k, logits.size(-1))
        cutoff = torch.topk(logits, k).values[-1]
        logits[logits < cutoff] = -float("inf")

    # 5. Top-p (nucleus)
    if top_p is not None and 0.0 < top_p < 1.0:
        sorted_logits, sorted_idx = torch.sort(logits, descending=True)
        cumulative = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
        remove = cumulative > top_p
        remove[1:] = remove[:-1].clone()       # keep the token that crosses top_p
        remove[0] = False
        logits[sorted_idx[remove]] = -float("inf")

    probs = F.softmax(logits, dim=-1)
    return int(torch.multinomial(probs, num_samples=1).item())


@torch.no_grad()
def generate_continuation(prompt, target_words=128, temperature=0.8,
                          top_k=50, top_p=0.95, max_new_tokens=None):
    """Return ONLY the continuation (prompt not included)."""

    seq2seq_model.eval()

    prompt_ids = tokenizer.encode(prompt)
    if len(prompt_ids) == 0:
        raise ValueError("Prompt produced zero tokens.")

    # Encoder sees the last SRC_LEN tokens of the prompt
    prompt_ids = prompt_ids[-SRC_LEN:]
    encoder_input = torch.tensor([prompt_ids], dtype=torch.long, device=DEVICE)

    # Encode once, reuse for every decoding step
    with autocast_context():
        encoder_output = seq2seq_model.encoder(encoder_input)

    if max_new_tokens is None:
        max_new_tokens = TGT_LEN - 1
    max_new_tokens = min(max_new_tokens, TGT_LEN - 1)

    generated = [BOS_ID]

    for _ in range(max_new_tokens):

        decoder_input = torch.tensor([generated], dtype=torch.long, device=DEVICE)

        with autocast_context():
            hidden = seq2seq_model.decoder(decoder_input, encoder_output)[:, -1, :]
            next_logits = seq2seq_model.lm_head(hidden)[0]

        next_token = sample_next_token(next_logits, generated[1:], temperature, top_k, top_p)

        if next_token == EOS_ID:
            break

        generated.append(next_token)

        # Word count check every 8 tokens (decoding every token is wasteful)
        if len(generated) % 8 == 0:
            text = tokenizer.decode(generated[1:])
            if len(text.split()) >= target_words:
                break

    text = tokenizer.decode([t for t in generated[1:] if t not in SPECIAL_IDS])
    return " ".join(text.split()[:target_words])


# ------------------------------------------------------------
# Quick checks
# ------------------------------------------------------------

assert banned_ngram_tokens([5, 6, 7, 8, 5, 6, 7], 4) == [8]
assert banned_ngram_tokens([5, 6, 7], 4) == []

_test = torch.zeros(VOCAB_SIZE)
_test[10], _test[11] = 2.0, -2.0
_test = apply_repetition_penalty(_test, [10, 11], 2.0, 64)
assert _test[10].item() == 1.0 and _test[11].item() == -4.0

print("REPETITION_PENALTY:", REPETITION_PENALTY, "| window:", REPETITION_WINDOW)
print("NO_REPEAT_NGRAM   :", NO_REPEAT_NGRAM, "| ALLOW_EARLY_EOS:", ALLOW_EARLY_EOS)
print("✅ Generation functions ready.")

REPETITION_PENALTY: 1.15 | window: 64
NO_REPEAT_NGRAM   : 12 | ALLOW_EARLY_EOS: False
✅ Generation functions ready.


### CELL 17 · Generation test

**In simple words:** Try a few prompts.

**How it works:** Output removed before publishing.

In [17]:
# ============================================================
# CELL 17 — GENERATION TEST
# ============================================================

TEST_PROMPTS = [
    "The issue involved in these petitions is as to whether "
    "the court has jurisdiction to consider the matter.",

    "In exercise of the powers conferred by section 3 of the Act, "
    "the State Government hereby",
]

torch.manual_seed(SEED)

for prompt in TEST_PROMPTS:

    start = time.time()
    continuation = generate_continuation(
        prompt,
        target_words=GEN_TARGET_WORDS,
        temperature=GEN_TEMPERATURE,
        top_k=GEN_TOP_K,
        top_p=GEN_TOP_P,
    )

    print("=" * 70)
    print("PROMPT:")
    print(prompt)
    print("\nCONTINUATION:")
    print(continuation)
    print(f"\n[{len(continuation.split())} words | {time.time() - start:.1f}s]")

    assert isinstance(continuation, str) and continuation.strip()
    assert continuation.strip() != prompt.strip()

print("=" * 70)
print("✅ Generation test completed.")

[output removed before publishing: this cell printed raw dataset / test text]
